## Looking inside — `exec`, `inspect`, `stats`

The container's running and you want to know what it's doing. A handful of tools open it up **without disturbing the main process.**

**`docker exec -it web sh`** — run a *new* command inside a running container. The single most-used debugging command: it drops you (or a script) into the container as a **sibling** of the main process, so exiting your shell leaves the container running.

```bash
docker exec -it web sh          # interactive shell
docker exec web ls /app         # one-shot command
docker exec -u root web bash    # as a specific user
```

**`docker attach web`** — different: it connects your terminal to the *existing* main process (PID 1), not a new one. Ctrl-C then sends `SIGINT` to PID 1 and usually **kills the container** — detach safely with `Ctrl-P Ctrl-Q`. Reach for `exec` almost always; `attach` only to watch PID 1's live stdout.

The read-only observers:

- **`docker inspect web`** — the full JSON state: image, command, env, mounts, network, restart policy. Use `--format '{{.State.Status}}'` to pull one field (module 01).
- **`docker stats`** — a live `top`-style stream of per-container CPU, memory, network, and block I/O. The fastest way to catch a memory leak or a CPU hog.
- **`docker top web`** — the processes running *inside* the container, from the host's view.
- **`docker diff web`** — every file added (`A`), changed (`C`), or deleted (`D`) in the container's writable layer versus its image. A neat way to see exactly what a running container has touched.

Together these are the debugging loop: `stats` to spot trouble, `exec` to get in, `inspect`/`top`/`diff` to see the details — all without a rebuild or a restart.